# Machine Learning 2025W — Exercise 1  
### Dataset Description and Exploration (breast cancer)

**Group Members:**  
- Full Name 1  
- Full Name 2  
- Full Name 3  

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
import ssl
import certifi
import itertools
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.metrics import classification_report
from sklearn.tree import plot_tree



## Load data

In [ ]:
file_path = '../datasets/184-702-tu-ml-2025w-breast-cancer/'
df= pd.read_csv(file_path+ "breast-cancer-diagnostic.shuf.lrn.csv")
df_kaggle_test= pd.read_csv(file_path+ "breast-cancer-diagnostic.shuf.tes.csv")
df=df.set_index("ID")
df.head()


In [ ]:
df.shape

### Basic Information

In [ ]:
df.info()
df.describe()
df.isna().sum()

In [ ]:
df_kaggle_test.info()
df_kaggle_test.describe()
df_kaggle_test.isna().sum()

### Target Variable Analysis

In [ ]:
target_col = 'class' 
sns.countplot(x=target_col, data=df)
plt.title('Breast cancer class - Training set')
plt.xlabel('Breast cancer class')
plt.show()

### Feature Exploration

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols].hist(figsize=(24, 16))
plt.suptitle("Numeric Feature Distributions")
plt.show()

# There are no categorical variables

# Preprocessing


## Target varaiable 

Since most models require numerical input, we will map the class to 0(no cancer) and 1(cancer).

In [ ]:
def target_to_num(x):
    if x==False:
        return 0
    else:
        return 1
    
df["class"]= df["class"].apply(target_to_num)

sns.countplot(x=target_col, data=df)
plt.title('Breast cancer class - Training set')
plt.xlabel('Breast cancer class')
plt.show()

## Input variables

### Check for outliers

In [ ]:
# Select numeric columns (excluding target variable)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'class' in numeric_cols:
    numeric_cols.remove('class') # class not included

print(f"Number of numeric columns to check: {len(numeric_cols)}")

# Define and run outlier detection function
def detect_outliers_iqr(df, columns):
    
    outlier_info = {}
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        outlier_count = len(outliers)
        outlier_percentage = (outlier_count / len(df)) * 100
        
        outlier_info[col] = {
            'count': outlier_count,
            'percentage': outlier_percentage,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound,
            'min': df[col].min(),
            'max': df[col].max(),
            'mean': df[col].mean(),
            'std': df[col].std()
        }
    
    return pd.DataFrame(outlier_info).T.sort_values('percentage', ascending=False)

# Run outlier detection
outlier_summary = detect_outliers_iqr(df, numeric_cols)
print("\nOutlier Summary:")
print(outlier_summary.head(10))

# Create the boxplots
n_cols = 5
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    df.boxplot(column=col, ax=axes[idx])
    outlier_pct = outlier_summary.loc[col, 'percentage']
    axes[idx].set_title(f'{col}', fontsize=9)
    axes[idx].tick_params(labelsize=8)


plt.tight_layout()
plt.show()

To decide how to handle the outliers, we first need to consider the background of the data. In this case it is medical data, where outliers can be very important for the diagnosis. Therefore, we will look if the outliers are data errors or just extreme values that should be kept in the dataset. 
In this case we flag data points as outlier if they are lower then Q1 - 1.5 * IQR or higher then Q3 + 1.5 * IQR.
In this data, the columns : areaStdErr, smoothnessStdErr, radiusStdErr, perimeterStdErr, fractalDimensionWorst, concavityStdErr, symmetryStdErr, areaMean, areaWorst, and symmetryWorst show outliers with a percentage of 11%-3 %.
We further looked at the plots and the min and max values of these columns and found that the outliers are not data errors but extreme values. Therefore, we will keep them in the dataset.

### Handle missing values

In [ ]:
print(df.isnull().sum())
print((df.astype(str)=="?").any())
print((df.astype(str)=="nan").any())

WE can see that there are no missing values in the dataset. This also alignes with the discriotion of the datatset provided.

In [ ]:
print(df_kaggle_test.isnull().sum())
print((df_kaggle_test.astype(str)=="?").any())
print((df_kaggle_test.astype(str)=="nan").any())

Also no missing values in  the test dataset.

## Split training data into train and test sets

In [ ]:
X = df.drop("class", axis=1)  # features
y = df["class"]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,       # 20% of data for testing
    random_state=42,     # ensures reproducibility
    stratify=y           # keeps same class proportions in train/test
)

X_train.head()

## Scale data

In [ ]:
#Scale training and test data
X_train_scaled=X_train.copy()
X_test_scaled= X_test.copy()
scaler = StandardScaler()
X_train_scaled[numeric_cols]= scaler.fit_transform(X_train_scaled[numeric_cols])
X_test_scaled[numeric_cols]= scaler.fit_transform(X_test_scaled[numeric_cols])
X_test_scaled.describe()

In [ ]:
print(numeric_cols)

# Model Training and Evaluation

We train three classifiers — Decision Tree, KNN, and Naive Bayes — and check how well they predict loan grades.

## Decision Tree Classifier

First we train decision tree clasifiers on multiple combinations of Hyperparameters (the result will be displayed as an dataframe) and inspect the results using specific plots 

In [ ]:

# Hyperparameter lists
max_depths = [3,5,10, None]
min_splits = [5,10, 20]
min_leafs = [3,5, 10]
criterions = ["gini", "entropy"]
weights = [None, "balanced"]

results = []

# Loop through all combinations
for depth, split, leaf, crit, weight in itertools.product(max_depths, min_splits, min_leafs, criterions, weights):
    
    clf = DecisionTreeClassifier(
        max_depth=depth,
        min_samples_split=split,
        min_samples_leaf=leaf,
        criterion=crit,
        class_weight=weight,
        random_state=42
    )
    clf.fit(X_train, y_train)
    
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:,1]  # probability for AUC
    
    results.append({
        "max_depth": depth,
        "min_samples_split": split,
        "min_samples_leaf": leaf,
        "criterion": crit,
        "class_weight": str(weight),
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average='macro'),
        "weighted_f1": f1_score(y_test, y_pred, average='weighted'),
        "auc": roc_auc_score(y_test, y_prob)
    })

# Convert to DataFrame
results_df = pd.DataFrame(results)
res_df_ordered= results_df.copy()
res_df_ordered.sort_values(by="macro_f1", ascending=False, inplace=True)
res_df_ordered = res_df_ordered.set_index(["class_weight","criterion","max_depth","min_samples_split","min_samples_leaf"])
res_df_ordered= res_df_ordered.sort_index()
res_df_ordered.head(50)

In [ ]:
# Plots for visualization

## Headmap for the differnt min/max values:
leaf_values = sorted(results_df["min_samples_leaf"].unique())

# Create a single figure with 1 row and N columns (one subplot per leaf value)
fig, axes = plt.subplots(1, len(leaf_values), figsize=(6 * len(leaf_values), 4), sharey=True)

for ax, leaf in zip(axes, leaf_values):
    subset = results_df[results_df["min_samples_leaf"] == leaf]
    heatmap_data = subset.pivot_table(
        values="macro_f1",
        index="max_depth",
        columns="min_samples_split"
    )
    
    sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="YlGnBu", ax=ax)
    ax.set_title(f"min_samples_leaf = {leaf}")
    ax.set_xlabel("min_samples_split")
    ax.set_ylabel("max_depth")

plt.suptitle("Macro F1 by max_depth and min_samples_split for different min_samples_leaf values", y=1.05)
plt.tight_layout()
plt.show()


## Bar chart to show the effect of differnt criterions
sns.barplot(x='class_weight', y='macro_f1', hue='criterion', data=results_df)
plt.title("Effect of class_weight and criterion on Macro F1")
plt.show()

## Plot to show the influnce of weighting the classes
sns.scatterplot(x='weighted_f1', y='macro_f1', hue='class_weight', style='criterion', data=results_df)
plt.title("Weighted F1 vs Macro F1 across Decision Tree variants (class_weight)")
plt.show()



We numerically search the best model dependent on the various evaluation metrics (macro_f1, weighted_f1, accuracy, auc).:

In [ ]:
# Choose the best combination of hyper paramters model:
hyperparams = ["max_depth","min_samples_split","min_samples_leaf","criterion","class_weight"]
# Pick the row with the highest Macro F1
best_model = results_df.loc[results_df['macro_f1'].idxmax()]

print("Optimal hyperparameter combination based on Macro F1:")
print(best_model[hyperparams])

# Pick the row with the highest weighted_f1
best_model = results_df.loc[results_df['weighted_f1'].idxmax()]

print("Optimal hyperparameter combination based on weighted_f1:")
print(best_model[hyperparams])

# Pick the row with the highest accuracy
best_model = results_df.loc[results_df['accuracy'].idxmax()]

print("Optimal hyperparameter combination based on accuracy:")
print(best_model[hyperparams])

# Pick the row with the highest auc
best_model = results_df.loc[results_df['auc'].idxmax()]

print("Optimal hyperparameter combination based on auc:")
print(best_model[hyperparams])


We fit the choosen best model and look what the most importnat features are:

In [ ]:
# Fit the evaluated optical model and get the most important attributes
clf_best = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=3,
    criterion="entropy",
    class_weight="balanced",
    random_state=42
)

clf_best.fit(X_train, y_train)

# Get feature importances
importances = clf_best.feature_importances_

# Combine with feature names
feature_importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print("Top important features:")
print(feature_importance_df.head(10)) 

We compare the evaluation metrics for the unscaled and scaled dataset:

In [ ]:
#Resulting evaulation values for the best selected model
test_pred = clf_best.predict(X_test)

print("Classification report for the UNSCALED data: ")
print(classification_report(y_test, test_pred))


# Apply the best model on the scaled data (shoulnt make any big difference)
clf_best_scaled = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=3,
    criterion="entropy",
    class_weight="balanced",
    random_state=42
)

clf_best_scaled.fit(X_train_scaled, y_train)

test_pred_scaled = clf_best_scaled.predict(X_test_scaled)

print("Classification report for the SCALED data: ")
print(classification_report(y_test, test_pred_scaled))

In [ ]:

plt.figure(figsize=(20, 10))
plot_tree(
    clf_best,
    filled=True,
    rounded=True,
    class_names=[str(cls) for cls in clf_best.classes_],
    feature_names=X_train.columns,
    fontsize=10
)
plt.title("Decision Tree (Best Model)")
plt.show()